# Sentiment Analysis with Transformer Models — BERT vs GPT-1
**Runtime:** GPU → T4  
**Tahmini süre:** ~50-60 dakika (tam dataset)

### Adımlar:
1. Kütüphane kurulumu
2. Dosya yükleme
3. GPU kontrolü
4. BERT fine-tuning (~30-35 dk)
5. GPT-1 fine-tuning (~20-25 dk)
6. Model karşılaştırması
7. Word raporu oluşturma
8. Dosyaları indirme

In [ ]:
# ── Adım 1: Kütüphane kurulumu ───────────────────────────────────────────────
!pip install -q torch transformers datasets scikit-learn tqdm python-docx
print('✓ Kütüphaneler kuruldu')

In [ ]:
# ── Adım 2: Dosya yükleme ────────────────────────────────────────────────────
# Şu 5 dosyayı birden seç ve yükle:
#   data_loader.py, train_bert.py, train_gpt.py,
#   compare_models.py, generate_report.py
from google.colab import files
uploaded = files.upload()

import os
os.makedirs('outputs/bert', exist_ok=True)
os.makedirs('outputs/gpt',  exist_ok=True)
print('Yüklenen dosyalar:', list(uploaded.keys()))

In [ ]:
# ── Adım 3: GPU kontrolü ─────────────────────────────────────────────────────
import torch
print('CUDA mevcut:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  GPU yok! Runtime > Change Runtime Type > T4 GPU seç')

In [ ]:
# ── Adım 4: BERT fine-tuning (~30-35 dakika) ─────────────────────────────────
# max_train_samples=5000 varsayılan — tüm dataset için None yapabilirsin
# ama 1 saatlik Colab limiti için 5000 yeterli ve sonuçlar iyi çıkıyor
%run train_bert.py
print('\n✓ BERT eğitimi tamamlandı')

In [ ]:
# ── Adım 5: GPT-1 fine-tuning (~20-25 dakika) ────────────────────────────────
%run train_gpt.py
print('\n✓ GPT-1 eğitimi tamamlandı')

In [ ]:
# ── Adım 6: Model karşılaştırması ────────────────────────────────────────────
%run compare_models.py

In [ ]:
# ── Adım 7: Word raporu oluşturma ─────────────────────────────────────────────
%run generate_report.py
import os
if os.path.exists('sentiment_analysis_report.docx'):
    print('✓ Rapor hazır: sentiment_analysis_report.docx')
else:
    print('⚠️  Rapor bulunamadı, generate_report.py çıktısını kontrol et')

In [ ]:
# ── Adım 8: Tüm çıktıları indir ──────────────────────────────────────────────
from google.colab import files
import os

for f in ['sentiment_analysis_report.docx',
          'outputs/bert/results.json',
          'outputs/gpt/results.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'✓ İndiriliyor: {f}')
    else:
        print(f'⚠️  Bulunamadı: {f}')

---
## ⏱️ Zaman az kalırsa — hızlı mod

Eğer Colab süresi dolmak üzereyse, Adım 4 ve 5 yerine aşağıdaki hücreyi çalıştır.
Daha az sample kullanır (~10 dakika) ama metrikler biraz düşer.

In [ ]:
# ── OPSİYONEL: Hızlı smoke-test (2000 sample, 2 epoch) ───────────────────────
# train_bert.py ve train_gpt.py içindeki load_imdb_data() çağrısını override et
import data_loader as dl
_orig = dl.load_imdb_data
dl.load_imdb_data = lambda **kw: _orig(max_train_samples=2000, **{k:v for k,v in kw.items() if k!='max_train_samples'})

import train_bert, train_gpt
train_bert.NUM_EPOCHS = 2
train_gpt.NUM_EPOCHS  = 2

train_bert.train()
train_gpt.train()